<a href="https://colab.research.google.com/github/AarushGoyal741/delhi-urban-expansion/blob/main/urban_expansion_NDBI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap -q

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='ee-your_project_name')

print("Earth Engine initialized successfully!")

In [ ]:
aoi = ee.Geometry.Rectangle([76.95, 28.45, 77.45, 28.88])

print("AOI defined successfully!")

In [ ]:
def mask_landsat_l2(image):
    qa_pixel = image.select('QA_PIXEL')
    qa_radsat = image.select('QA_RADSAT')

    mask = (
        qa_pixel.bitwiseAnd(1 << 4).eq(0)   # cloud shadow
        .And(qa_pixel.bitwiseAnd(1 << 3).eq(0))  # cloud
        .And(qa_pixel.bitwiseAnd(1 << 2).eq(0))  # cirrus
        .And(qa_pixel.bitwiseAnd(1 << 5).eq(0))  # snow
        .And(qa_radsat.eq(0))
    )

    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, overwrite=True).updateMask(mask)

print("Masking function ready!")

In [ ]:
# Landsat 5 for 2010
img_2010 = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
    .filterBounds(aoi)
    .filterDate('2010-03-01', '2010-05-31')
    .map(mask_landsat_l2)
    .median()
    .clip(aoi)
)

# Landsat 8 for 2023
img_2023 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(aoi)
    .filterDate('2023-03-01', '2023-05-31')
    .map(mask_landsat_l2)
    .median()
    .clip(aoi)
)

print("Images loaded successfully!")

In [ ]:
def indices_landsat5(img):
    ndbi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDBI')
    ndvi = img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    mndwi = img.normalizedDifference(['SR_B2', 'SR_B5']).rename('MNDWI')
    return ndbi, ndvi, mndwi

def indices_landsat8(img):
    ndbi = img.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return ndbi, ndvi, mndwi

print("Index functions ready!")

In [ ]:
ndbi_2010, ndvi_2010, mndwi_2010 = indices_landsat5(img_2010)
ndbi_2023, ndvi_2023, mndwi_2023 = indices_landsat8(img_2023)

print("Indices calculated successfully!")

In [ ]:
def get_percentile(image, region, percentile):
    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([percentile]),
        geometry=region,
        scale=30,
        maxPixels=1e13
    )
    key = ee.String(stats.keys().get(0))
    return ee.Number(stats.get(key))

print("Percentile helper ready!")

In [ ]:
threshold_2023 = get_percentile(ndbi_2023, aoi, 65)

builtup_2023 = (
    ndbi_2023.gt(threshold_2023)
    .And(ndvi_2023.lt(0.30))
    .And(mndwi_2023.lt(0.0))
    .selfMask()
    .rename('Builtup_2023')
)

builtup_2023 = builtup_2023.focal_max(1).focal_min(1)
connected_2023 = builtup_2023.connectedPixelCount(50, True)
builtup_2023 = builtup_2023.updateMask(connected_2023.gte(3))

print("2023 built-up mask created!")
print("Threshold 2023:", threshold_2023.getInfo())

In [ ]:
built_score_2010 = (
    ndbi_2010
    .subtract(ndvi_2010)
    .subtract(mndwi_2010)
    .rename('BUILT_SCORE')
)

print("Built Score 2010 created!")

In [ ]:
lower_threshold_2010 = get_percentile(built_score_2010, aoi, 20)
upper_threshold_2010 = get_percentile(built_score_2010, aoi, 30)

print("Lower threshold 2010:", lower_threshold_2010.getInfo())
print("Upper threshold 2010:", upper_threshold_2010.getInfo())

In [ ]:
builtup_2010 = (
    built_score_2010.gte(lower_threshold_2010)
    .And(built_score_2010.lte(upper_threshold_2010))
    .And(ndvi_2010.lt(0.35))
    .And(mndwi_2010.lt(0.0))
    .selfMask()
    .rename('Builtup_2010')
)

print("2010 built-up mask created using score range!")

In [ ]:
builtup_2010 = builtup_2010.focal_max(1).focal_min(1)
connected_2010 = builtup_2010.connectedPixelCount(50, True)
builtup_2010 = builtup_2010.updateMask(connected_2010.gte(3))

print("2010 cleanup applied!")

In [ ]:
def calculate_area_sqkm(binary_image, region):
    area_img = binary_image.multiply(ee.Image.pixelArea())
    stats = area_img.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=30,
        maxPixels=1e13
    )
    band_name = binary_image.bandNames().get(0)
    area_m2 = ee.Number(stats.get(band_name))
    return area_m2.divide(1e6)

print("Area function ready!")

In [ ]:
area_2010 = calculate_area_sqkm(builtup_2010, aoi)
area_2023 = calculate_area_sqkm(builtup_2023, aoi)
increase = area_2023.subtract(area_2010)

print("Built-up area 2010 (sq. km):", area_2010.getInfo())
print("Built-up area 2023 (sq. km):", area_2023.getInfo())
print("Increase in built-up area (sq. km):", increase.getInfo())

In [ ]:
change = builtup_2023.unmask(0).subtract(builtup_2010.unmask(0)).rename('Change')

print("Change layer created!")

In [ ]:
Map = geemap.Map(center=[28.67, 77.20], zoom=9)

score_vis = {
    'min': -1,
    'max': 1,
    'palette': ['purple', 'white', 'red']
}

ndbi_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['blue', 'white', 'brown']
}

builtup_2010_vis = {'palette': ['red']}
builtup_2023_vis = {'palette': ['blue']}

change_vis = {
    'min': -1,
    'max': 1,
    'palette': ['yellow', 'white', 'green']
}

Map.addLayer(built_score_2010, score_vis, 'Built Score 2010')
Map.addLayer(ndbi_2023, ndbi_vis, 'NDBI 2023')
Map.addLayer(builtup_2010, builtup_2010_vis, 'Built-up 2010')
Map.addLayer(builtup_2023, builtup_2023_vis, 'Built-up 2023')
Map.addLayer(change, change_vis, 'Urban Expansion')
Map.addLayer(aoi, {}, 'AOI')

Map

In [ ]:
results = pd.DataFrame({
    'Year': ['2010', '2023'],
    'Built-up Area (sq. km)': [area_2010.getInfo(), area_2023.getInfo()]
})

results['Change from 2010 (sq. km)'] = [0, increase.getInfo()]

results

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(['2010', '2023'], [area_2010.getInfo(), area_2023.getInfo()])
plt.xlabel('Year')
plt.ylabel('Built-up Area (sq. km)')
plt.title('Built-up Area Comparison')
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

os.makedirs('outputs', exist_ok=True)
print("✅ Ready!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#0d1b2a')

rect = Rectangle((76.95, 28.45), 0.5, 0.43,
                 linewidth=2.5, edgecolor='red', facecolor='#1a3a5c', alpha=0.4)
ax.add_patch(rect)

locations = {
    'New Delhi (Core)': (77.21, 28.63),
    'Gurgaon':          (77.03, 28.46),
    'Noida':            (77.33, 28.57),
    'Rohini':           (77.07, 28.70),
    'Faridabad':        (77.31, 28.47),
    'Bahadurgarh':      (76.92, 28.69),
}
for name, (lon, lat) in locations.items():
    ax.plot(lon, lat, 'o', color='#FFD700', markersize=7, zorder=5)
    ax.annotate(name, (lon, lat), textcoords='offset points',
                xytext=(6, 4), color='white', fontsize=9,
                fontweight='bold')

# Yamuna river approximate line
yamuna_lon = [77.21, 77.23, 77.25, 77.27, 77.29, 77.30]
yamuna_lat = [28.75, 28.70, 28.63, 28.58, 28.53, 28.48]
ax.plot(yamuna_lon, yamuna_lat, '-', color='#00BFFF',
        linewidth=2.5, label='Yamuna River', zorder=4)

ax.set_xlim(76.82, 77.58)
ax.set_ylim(28.35, 29.00)
ax.set_xlabel('Longitude (°E)', color='white', fontsize=11)
ax.set_ylabel('Latitude (°N)', color='white', fontsize=11)
ax.tick_params(colors='white')
ax.spines[:].set_color('#555')
ax.grid(color='#333', linestyle='--', linewidth=0.5, alpha=0.5)

ax.set_title('Study Area: Delhi NCR\nArea of Interest (AOI)',
             color='white', fontsize=14, fontweight='bold', pad=15)

ax.text(77.195, 28.885,
        'AOI Coordinates:\n76.95°E – 77.45°E\n28.45°N – 28.88°N',
        color='red', fontsize=9, ha='center',
        bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                  alpha=0.8, edgecolor='red'))

legend_elements = [
    mpatches.Patch(facecolor='#1a3a5c', edgecolor='red',
                   label='Area of Interest (AOI)'),
    plt.Line2D([0],[0], color='#00BFFF',
               linewidth=2, label='Yamuna River'),
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor='#FFD700',
               markersize=8, label='Key Locations'),
]
ax.legend(handles=legend_elements, loc='lower right',
          framealpha=0.4, labelcolor='white',
          facecolor='#1a1a2e', edgecolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/aoi_map.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("✅ AOI map saved!")

In [ ]:
def get_arr(image, band, aoi, scale=150):
    arr = geemap.ee_to_numpy(
        image.select(band), region=aoi, scale=scale)
    if arr is None:
        return None
    a = arr[:,:,0].astype(float)
    a[a < -1] = np.nan
    return a

print("Downloading NDBI arrays...")
ndbi_arr_2010 = get_arr(ndbi_2010, 'NDBI', aoi)
ndbi_arr_2023 = get_arr(ndbi_2023, 'NDBI', aoi)
print("✅ Downloaded!")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#1a1a2e')

for ax, arr, year, sat in zip(
        axes,
        [ndbi_arr_2010, ndbi_arr_2023],
        ['2010', '2023'],
        ['Landsat 5', 'Landsat 8']):
    ax.set_facecolor('#1a1a2e')
    im = ax.imshow(arr, cmap='RdYlGn_r', vmin=-0.4, vmax=0.4)
    ax.set_title(f'NDBI Raster — {year}\n({sat}, Jan–May)',
                 color='white', fontsize=13,
                 fontweight='bold', pad=10)
    ax.axis('off')
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.04)
    cbar.set_label('NDBI Value', color='white', fontsize=10)
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
    ax.text(0.02, 0.02,
            'High NDBI (red) = Built-up\nLow NDBI (green) = Vegetation',
            transform=ax.transAxes, color='white',
            fontsize=8, verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                      alpha=0.7, edgecolor='gray'))

fig.suptitle(
    'Normalized Difference Built-up Index (NDBI)\n'
    'Delhi NCR — 2010 vs 2023',
    color='white', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/ndbi_raster_comparison.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ NDBI raster map saved!")

In [ ]:
print("Downloading built-up arrays...")
bu_arr_2010 = get_arr(builtup_2010, 'Builtup_2010', aoi)
bu_arr_2023 = get_arr(builtup_2023, 'Builtup_2023', aoi)
print("✅ Downloaded!")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#1a1a2e')

data = [
    (bu_arr_2010, '2010', 'Landsat 5', '#FF8C00', '953.95 km²'),
    (bu_arr_2023, '2023', 'Landsat 8', '#FF2020', '1408.89 km²'),
]

for ax, (arr, year, sat, color, area) in zip(axes, data):
    ax.set_facecolor('#0a0a1a')
    bg = np.zeros((*arr.shape, 4))
    ax.imshow(bg)
    rgba = np.zeros((*arr.shape, 4))
    mask = (arr == 1)
    import matplotlib.colors as mcolors
    rgba[mask]  = mcolors.to_rgba(color)
    rgba[~mask, 3] = 0
    ax.imshow(rgba)
    ax.set_title(f'Built-up Area — {year}\n({sat}, NDBI > threshold)',
                 color='white', fontsize=13,
                 fontweight='bold', pad=10)
    ax.axis('off')
    ax.text(0.98, 0.02, f'Built-up: {area}',
            transform=ax.transAxes, color='white',
            fontsize=11, ha='right', va='bottom',
            fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                      alpha=0.8, edgecolor=color))

fig.suptitle(
    'Built-up Area Extraction using NDBI Threshold\n'
    'Delhi NCR — 2010 vs 2023',
    color='white', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/builtup_maps.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Built-up maps saved!")

In [ ]:
print("Downloading change array...")
change_arr = get_arr(
    builtup_2023.unmask(0).subtract(builtup_2010.unmask(0)).rename('Change'),
    'Change', aoi)
print("✅ Downloaded!")

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#0a0a1a')

import matplotlib.colors as mcolors
change_rgba = np.zeros((*change_arr.shape, 4))

# No change — dark background
no_change = (change_arr == 0)
change_rgba[no_change] = [0.05, 0.05, 0.1, 1.0]

# New urban (0→1) — bright red
new_urban_mask = (change_arr == 1)
change_rgba[new_urban_mask] = mcolors.to_rgba('#FF2020')



ax.imshow(change_rgba)
ax.set_title('Urban Expansion Change Detection\nDelhi NCR (2010 → 2023)',
             color='white', fontsize=15, fontweight='bold', pad=15)
ax.axis('off')

legend_elements = [
    mpatches.Patch(color='#0d0d1a', label='No Change',
                   edgecolor='gray'),
    mpatches.Patch(color='#FF2020',
                   label=f'New Urban Area (+454.94 km²)'),

]
ax.legend(handles=legend_elements, loc='lower left',
          fontsize=11, framealpha=0.5,
          labelcolor='white', facecolor='#1a1a2e',
          edgecolor='white')

ax.text(0.98, 0.98,
        'Built-up 2010: 953.95 km²\n'
        'Built-up 2023: 1408.89 km²\n'
        'New Urban: +454.94 km²\n'
        'Growth: +47.7% in 13 years',
        transform=ax.transAxes,
        color='white', fontsize=11,
        va='top', ha='right',
        bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                  alpha=0.8, edgecolor='white'))

plt.tight_layout()
plt.savefig('outputs/change_detection_map.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Change detection map saved!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#1a1a2e')

for ax in axes:
    ax.set_facecolor('#0f3460')
    ax.spines[:].set_color('#555')
    ax.tick_params(colors='white', labelsize=11)
    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.title.set_color('white')

# Bar chart
years  = ['2010', '2023']
areas  = [953.95, 1408.89]
colors = ['#FF8C00', '#FF2020']

bars = axes[0].bar(years, areas, color=colors,
                   width=0.5, edgecolor='white', linewidth=0.8)
axes[0].set_title('Built-up Area Comparison',
                  fontweight='bold', fontsize=13)
axes[0].set_ylabel('Area (km²)', fontsize=11)
axes[0].set_ylim(0, 1700)

for bar, area in zip(bars, areas):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 25,
        f'{area} km²',
        ha='center', color='white',
        fontweight='bold', fontsize=11)

# Growth pie
sizes  = [953.95, 454.94]
labels = ['Existing Urban\n(2010)\n953.95 km²',
          'New Urban\n(2010→2023)\n454.94 km²']
colors_pie = ['#FF8C00', '#FF2020']

wedges, texts, autotexts = axes[1].pie(
    sizes, labels=labels, colors=colors_pie,
    explode=(0, 0.08), autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white', 'fontsize': 10})
for at in autotexts:
    at.set_color('white')
    at.set_fontweight('bold')

axes[1].set_title('Urban Growth Composition (2023)',
                  fontweight='bold', fontsize=13)

fig.suptitle(
    'Delhi NCR Urban Expansion Statistics (2010–2023)\n'
    '+47.7% growth | +3.67% per year | +454.94 km² added',
    color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/stats_chart.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Stats chart saved!")